# Consolidated SDC Sweep Notebook

PE split -> Mock sweeps (n=50, all 4 fault types) -> 5-key collection -> real-channel sweeps (n=20, placeholder+asymptotic, all 4 fault types, key0) -> multi-key sweeps (5 keys, toeplitz/final-key) -> completeness check + plots.

No finite_key data collected here -- placeholder and asymptotic only.

In [1]:
import sys, json, re, time
from pathlib import Path
import numpy as np
import pandas as pd
from galois import GF2
from randextract import ToeplitzHashing

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))

import deploy_fabric as deploy
from qne.cascade import Key, ORIGINAL, Reconciliation, MockClassicalSession, SDCFaultInjector
from qne.cascade.key import key_from_sifted_json
from qne.cascade.sweep_utils import (
    run_condition_sweep, summarize_outcomes,
    run_real_channel_trial, run_real_channel_reconciliation_trial,
    collect_key_pairs,
)

SLICE_NAME = 'qfabric-bb84-2'
fablib = deploy.get_fablib()
slice_obj = fablib.get_slice(name=SLICE_NAME)
slice_obj.show()

alice = slice_obj.get_node("alice")
bob = slice_obj.get_node("bob")
bob_ip = "10.10.1.2"

print("Setup complete: slice, nodes, imports ready")


Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
CEPH Manager,https://ceph-mgr.fabric-testbed.net
Token File,/home/fabric/work/fabric_config/id_token.json
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
Bastion Host,bastion.fabric-testbed.net
Bastion Username,audreyf_0000527467
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub


User: audreyf@illinois.edu bastion key is valid!
Configuration is valid


ID,900e9b70-47f3-4394-a831-e522acb47878
Name,qfabric-bb84-2
Lease Expiration (UTC),2026-08-25 21:17:14 +0000
Lease Start (UTC),2026-08-11 21:17:14 +0000
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
State,StableOK
Email,audreyf@illinois.edu
UserId,8616dd8c-5b61-45db-84bb-791c21e82a89


Setup complete: slice, nodes, imports ready


## 1. Parameter-estimation split (key0)

Splits the raw sifted key into a disjoint PE sample (k bits, used only to estimate QBER) and a generation string (n bits, used for EC/PA). Reuses the existing split if already computed, rather than re-splitting (which would incorrectly sample a PE subset from already-split generation bits).

In [3]:
from qne.cascade.parameter_estimation import split_pe_and_generation
import json as _json

RAW_ALICE = PROJECT_DIR / "results" / "fabric_alice_sifted_bits.json"
RAW_BOB = PROJECT_DIR / "results" / "fabric_bob_sifted_bits.json"
GEN_ALICE = PROJECT_DIR / "results" / "fabric_alice_sifted_bits_genonly.json"
GEN_BOB = PROJECT_DIR / "results" / "fabric_bob_sifted_bits_genonly.json"
META_PATH = PROJECT_DIR / "results" / "fabric_key0_pe_meta.json"

alice_key_full, alice_indices_full = key_from_sifted_json(str(RAW_ALICE), "alice_bits")
bob_key_full, bob_indices_full = key_from_sifted_json(str(RAW_BOB), "bob_bits")
assert alice_indices_full == bob_indices_full, "alice and bob's matching indices don't match"

if GEN_ALICE.exists() and GEN_BOB.exists() and META_PATH.exists():
    # Already split in a previous run -- reuse it rather than re-splitting
    # (re-splitting an already-split key would sample a PE subset from
    # what's actually generation-only bits, which is wrong).
    alice_key, alice_indices = key_from_sifted_json(str(GEN_ALICE), "alice_bits")
    bob_key, bob_indices = key_from_sifted_json(str(GEN_BOB), "bob_bits")
    meta = _json.loads(META_PATH.read_text())
    k_pe, real_qber = meta["k"], meta["qber"]
    print(f"Reusing existing PE split: k={k_pe}, real_qber={real_qber:.4f}")
else:
    split = split_pe_and_generation(alice_key_full, bob_key_full, sample_fraction=0.1, seed=999001)
    alice_key, bob_key = split["alice_gen"], split["bob_gen"]
    k_pe, real_qber = split["k"], split["qber"]

    gen_indices = split["gen_indices"]
    gen_matching_indices = [alice_indices_full[idx] for idx in gen_indices]
    GEN_ALICE.write_text(_json.dumps({"alice_bits": alice_key.bits.tolist(), "matching_indices": gen_matching_indices}))
    GEN_BOB.write_text(_json.dumps({"bob_bits": bob_key.bits.tolist(), "matching_indices": gen_matching_indices}))
    META_PATH.write_text(_json.dumps({"k": k_pe, "qber": real_qber, "m": split["m"], "n": split["n"]}))

    print(f"m={split['m']} sifted, k={k_pe} PE sample, n={split['n']} generation bits, "
          f"real_qber (from disjoint PE sample) = {real_qber:.4f}")

# Re-sync both remote nodes to this exact (PE-split, generation-only) key pair.
alice.upload_file(str(GEN_ALICE), "qfabric/results/alice_sifted_bits.json")
bob.upload_file(str(GEN_ALICE), "qfabric/results/alice_sifted_bits.json")
bob.upload_file(str(GEN_BOB), "qfabric/results/bob_sifted_bits.json")
print("Remote nodes synced to PE-split generation-only key")


Reusing existing PE split: k=388, real_qber=0.0077
Remote nodes synced to PE-split generation-only key


## 2. Mock dose-response (n=50/point), all four fault types

This is the larger, n=50 version -- supersedes any smaller n=10 Mock sweep. Verify-digest split uses the **placeholder** length (not finite-key), consistent with the rest of this notebook.

In [ ]:
"""
Scaled-up Mock dose-response: Toeplitz, final-key, reconciliation, and
verify-digest, all at n=50 per probability point.
"""
probs_th = [0.5, 0.3, 0.1, 0.05, 0.03, 0.01, 0.003, 0.001]
probs_recon = [0.3, 0.1, 0.03, 0.01, 0.003, 0.001]
all_dfs, summary_rows = {}, []

for fault_type, probs, kwarg_name in [("toeplitz", probs_th, "toeplitz_prob"),
                                        ("final_key", probs_th, "final_key_prob"),
                                        ("reconciliation", probs_recon, "reconciliation_prob")]:
    for prob in probs:
        label = f"{fault_type}_{prob}"
        print(f"Running {label} (n=50)...")
        df_cond = run_condition_sweep(alice_key, bob_key, real_qber, label, n_runs=50, **{kwarg_name: prob})
        df_cond["fault_type"] = fault_type
        df_cond["prob"] = prob
        all_dfs[label] = df_cond
        summary_rows.append(summarize_outcomes(df_cond, label))

# --- Verify-digest sweep: split uses the PLACEHOLDER length (not finite-key) ---
n_key0 = alice_key.get_nr_bits()
ell = ToeplitzHashing.calculate_length(
    extractor_type="quantum", input_length=n_key0,
    relative_source_entropy=0.5, error_bound=1e-6,
)
t_placeholder = max(1, int(np.ceil(-np.log2(1e-10))))  # same fallback t the driver uses in placeholder mode
digest_length = max(1, int(np.ceil(t_placeholder)))
t_verify = max(2 * digest_length, 32)
print(f"\nVerify-digest split (placeholder length): ell={ell}, t_verify={t_verify}, digest_length={digest_length}")

for prob in probs_th:
    label = f"verify_digest_{prob}"
    print(f"Running {label} (n=50)...")
    df_cond = run_condition_sweep(alice_key, bob_key, real_qber, label, n_runs=50,
                                     verify_digest_prob=prob, ell=ell, t_verify=t_verify,
                                     digest_length=digest_length)
    df_cond["fault_type"] = "verify_digest"
    df_cond["prob"] = prob
    all_dfs[label] = df_cond
    summary_rows.append(summarize_outcomes(df_cond, label))

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

full_df = pd.concat(all_dfs.values(), ignore_index=True)
full_df.to_csv(str(PROJECT_DIR / "results" / "sdc_mock_doseresponse_n50.csv"), index=False)
print(f"Saved {len(full_df)} rows -> sdc_mock_doseresponse_n50.csv")


## 3. Collect 5 real key pairs (FABRIC)

Unique to this section -- not duplicated elsewhere. Saves per-key metadata (k_pe, n_bits, qber) internally to `key_pairs_metadata.csv`.

In [ ]:
key_pairs_df = collect_key_pairs(deploy, slice_obj, alice, bob, bob_ip, PROJECT_DIR, n_keys=5)
# collect_key_pairs already saves this internally to key_pairs_metadata.csv
print(key_pairs_df)


## 4. Real-channel sweeps, n=20/point, key0, placeholder + asymptotic length modes

All four fault types. Each cell saves incrementally where noted, so a disconnect never loses more than one trial.

In [ ]:
"""
Toeplitz-matrix faults, real channel, n=20 per probability, both length modes.
"""
length_modes = ["placeholder", "asymptotic"]
rows = []

for length_mode in length_modes:
    for prob in probs_th:
        print(f"\n=== toeplitz prob={prob}, length_mode={length_mode} (n=20) ===")
        for run in range(20):
            seed = 42 + run
            result = run_real_channel_trial(bob, alice, bob_ip, real_qber, run, seed, k=k_pe,
                                               length_mode=length_mode, toeplitz_prob=prob)
            result["fault_type"] = "toeplitz"
            result["prob"] = prob
            result["length_mode"] = length_mode
            rows.append(result)
            if run % 5 == 0:
                print(f"  run {run}: keys_match={result.get('keys_match')}, "
                      f"secure_key_length={result.get('secure_key_length')}")

df_toeplitz_n20 = pd.DataFrame(rows)
df_toeplitz_n20.to_csv(str(PROJECT_DIR / "results" / "sdc_realchannel_toeplitz_n20_all_modes.csv"), index=False)
print(f"\nSaved {len(df_toeplitz_n20)} rows -> sdc_realchannel_toeplitz_n20_all_modes.csv")


In [ ]:
"""
Final-key faults, real channel, n=20 per probability, both length modes.
Saves incrementally.
"""
import os

final_key_output_path = str(PROJECT_DIR / "results" / "sdc_realchannel_finalkey_n20_all_modes.csv")

if os.path.exists(final_key_output_path):
    df_existing = pd.read_csv(final_key_output_path)
    completed = set(zip(df_existing["length_mode"], df_existing["prob"], df_existing["run"]))
    print(f"Resuming: {len(completed)} trials already done")
else:
    completed = set()
    print("Starting fresh")

header_written = os.path.exists(final_key_output_path)

for length_mode in ["placeholder", "asymptotic"]:
    for prob in probs_th:
        for run in range(20):
            if (length_mode, prob, run) in completed:
                continue

            seed = 42 + run
            result = run_real_channel_trial(bob, alice, bob_ip, real_qber, run, seed, k=k_pe,
                                               length_mode=length_mode, final_key_prob=prob)
            result["fault_type"] = "final_key"
            result["prob"] = prob
            result["length_mode"] = length_mode

            row_df = pd.DataFrame([result])
            row_df.to_csv(final_key_output_path, mode="a", header=not header_written, index=False)
            header_written = True

            print(f"  mode={length_mode}, prob={prob}, run={run}: "
                  f"keys_match={result.get('keys_match')}, "
                  f"verification_passed={result.get('verification_passed')} [saved]")

print("\nFinal-key sweep complete.")
print(pd.read_csv(final_key_output_path).groupby(["length_mode", "prob"]).size())


In [ ]:
"""
Reconciliation-state faults, real channel, n=20 per probability, both length modes.
Saves incrementally.
"""
recon_output_path = str(PROJECT_DIR / "results" / "sdc_realchannel_reconciliation_n20_all_modes.csv")

if os.path.exists(recon_output_path):
    df_existing = pd.read_csv(recon_output_path)
    completed = set(zip(df_existing["length_mode"], df_existing["prob"], df_existing["run"]))
    print(f"Resuming: {len(completed)} trials already done")
else:
    completed = set()
    print("Starting fresh")

header_written = os.path.exists(recon_output_path)

for length_mode in ["placeholder", "asymptotic"]:
    for prob in probs_recon:
        for run in range(20):
            if (length_mode, prob, run) in completed:
                continue

            seed = 42 + run
            result = run_real_channel_reconciliation_trial(bob, alice, bob_ip, real_qber, run, seed,
                                                               reconciliation_prob=prob, k=k_pe,
                                                               length_mode=length_mode)
            result["fault_type"] = "reconciliation"
            result["prob"] = prob
            result["length_mode"] = length_mode

            row_df = pd.DataFrame([result])
            row_df.to_csv(recon_output_path, mode="a", header=not header_written, index=False)
            header_written = True

            print(f"  mode={length_mode}, prob={prob}, run={run}: "
                  f"non_convergent={result.get('non_convergent')} [saved]")

print("\nReconciliation sweep complete.")
print(pd.read_csv(recon_output_path).groupby(["length_mode", "prob"]).size())


In [ ]:
"""
Verification-digest faults, real channel, n=20 per probability, both length modes.
Saves incrementally.
"""
verify_digest_output_path = str(PROJECT_DIR / "results" / "sdc_realchannel_verifydigest_n20_all_modes.csv")

if os.path.exists(verify_digest_output_path):
    df_existing = pd.read_csv(verify_digest_output_path)
    completed = set(zip(df_existing["length_mode"], df_existing["prob"], df_existing["run"]))
    print(f"Resuming: {len(completed)} trials already done")
else:
    completed = set()
    print("Starting fresh")

header_written = os.path.exists(verify_digest_output_path)

for length_mode in ["placeholder", "asymptotic"]:
    for prob in probs_th:
        for run in range(20):
            if (length_mode, prob, run) in completed:
                continue

            seed = 42 + run
            result = run_real_channel_trial(bob, alice, bob_ip, real_qber, run, seed, k=k_pe,
                                               length_mode=length_mode, verify_digest_prob=prob)
            result["fault_type"] = "verify_digest"
            result["prob"] = prob
            result["length_mode"] = length_mode

            row_df = pd.DataFrame([result])
            row_df.to_csv(verify_digest_output_path, mode="a", header=not header_written, index=False)
            header_written = True

            print(f"  mode={length_mode}, prob={prob}, run={run}: "
                  f"keys_match={result.get('keys_match')}, "
                  f"verification_passed={result.get('verification_passed')} [saved]")

print("\nVerification-digest sweep complete.")
print(pd.read_csv(verify_digest_output_path).groupby(["length_mode", "prob"]).size())


## 5. Multi-key sweeps (5 keys), asymptotic length mode

Unique to this section -- tests across the 5 different collected keys (different QBER/n), not just key0. `k`/`qber` for each key are pulled from `key_pairs_df`, never mixed with key0's `k_pe`/`real_qber`.

In [ ]:
key_files = [(f"alice_sifted_bits_key{i}.json", f"bob_sifted_bits_key{i}.json") for i in range(5)]
conditions = [("toeplitz_only", {"toeplitz_prob": 0.5}), ("final_key_only", {"final_key_prob": 0.5})]

all_rows = []
for key_idx, (a_file, b_file) in enumerate(key_files):
    meta_row = key_pairs_df.iloc[key_idx]
    k_i, qber_i = int(meta_row["k_pe"]), float(meta_row["qber"])

    alice.upload_file(str(PROJECT_DIR / "results" / a_file), "qfabric/results/alice_sifted_bits.json")
    bob.upload_file(str(PROJECT_DIR / "results" / a_file), "qfabric/results/alice_sifted_bits.json")
    bob.upload_file(str(PROJECT_DIR / "results" / b_file), "qfabric/results/bob_sifted_bits.json")

    print(f"\n=== key{key_idx}: {int(meta_row['n_bits'])} generation bits, k={k_i} PE sample, "
          f"QBER (PE sample) = {qber_i:.4f} (asymptotic length) ===")

    for label, kwargs in conditions:
        print(f"  Running {label}...")
        for run in range(5):
            seed = 42 + run
            result = run_real_channel_trial(bob, alice, bob_ip, qber_i, run, seed, k=k_i,
                                               length_mode="asymptotic", **kwargs)
            result["condition"] = label
            result["key_index"] = key_idx
            result["qber"] = qber_i
            all_rows.append(result)
            print(f"    run {run}: keys_match={result.get('keys_match')}, "
                  f"faults_fired={result.get('faults_fired')}")

df_multikey = pd.DataFrame(all_rows)
df_multikey.to_csv(str(PROJECT_DIR / "results" / "sdc_real_channel_toeplitz_finalkey_multikey_asymptotic.csv"), index=False)
print(f"\nSaved {len(df_multikey)} rows.")


In [ ]:
key_files = [(f"alice_sifted_bits_key{i}.json", f"bob_sifted_bits_key{i}.json") for i in range(5)]
probs = [0.5, 0.3, 0.1, 0.05, 0.01]

all_rows = []
for key_idx, (a_file, b_file) in enumerate(key_files):
    meta_row = key_pairs_df.iloc[key_idx]
    k_i, qber_i = int(meta_row["k_pe"]), float(meta_row["qber"])

    alice.upload_file(str(PROJECT_DIR / "results" / a_file), "qfabric/results/alice_sifted_bits.json")
    bob.upload_file(str(PROJECT_DIR / "results" / a_file), "qfabric/results/alice_sifted_bits.json")
    bob.upload_file(str(PROJECT_DIR / "results" / b_file), "qfabric/results/bob_sifted_bits.json")
    print(f"\n=== key{key_idx}: QBER (PE sample) = {qber_i:.4f} (asymptotic length) ===")

    for prob in probs:
        for fault_type, kwargs in [("toeplitz", {"toeplitz_prob": prob}), ("final_key", {"final_key_prob": prob})]:
            print(f"  {fault_type} prob={prob}...")
            for run in range(3):
                seed = 42 + run
                result = run_real_channel_trial(bob, alice, bob_ip, qber_i, run, seed, k=k_i,
                                                   length_mode="asymptotic", **kwargs)
                result["fault_type"] = fault_type
                result["prob"] = prob
                result["key_index"] = key_idx
                result["qber"] = qber_i
                all_rows.append(result)

df_doseresponse = pd.DataFrame(all_rows)
df_doseresponse.to_csv(str(PROJECT_DIR / "results" / "sdc_real_channel_toeplitz_finalkey_doseresponse_multikey_asymptotic.csv"), index=False)
print(f"\nSaved {len(df_doseresponse)} rows.")


In [ ]:
"""
Multi-key reconciliation dose-response (5 keys, asymptotic length),
using each key's own (k, qber) -- never mixed with key0's real_qber.
"""
recon_output_path_multikey = str(PROJECT_DIR / "results" / "sdc_real_channel_reconciliation_multikey_asymptotic.csv")

if os.path.exists(recon_output_path_multikey):
    df_existing = pd.read_csv(recon_output_path_multikey)
    completed = set(zip(df_existing["key_index"], df_existing["reconciliation_prob"], df_existing["run"]))
    print(f"Resuming: {len(completed)} trials already done")
else:
    completed = set()
    print("Starting fresh")

header_written = os.path.exists(recon_output_path_multikey)
recon_probs_multikey = [0.3, 0.1, 0.03, 0.01, 0.003, 0.001]

for key_idx, (a_file, b_file) in enumerate(key_files):
    meta_row = key_pairs_df.iloc[key_idx]
    k_i, qber_i = int(meta_row["k_pe"]), float(meta_row["qber"])

    alice.upload_file(str(PROJECT_DIR / "results" / a_file), "qfabric/results/alice_sifted_bits.json")
    bob.upload_file(str(PROJECT_DIR / "results" / a_file), "qfabric/results/alice_sifted_bits.json")
    bob.upload_file(str(PROJECT_DIR / "results" / b_file), "qfabric/results/bob_sifted_bits.json")

    for prob in recon_probs_multikey:
        for run in range(5):
            if (key_idx, prob, run) in completed:
                continue

            seed = 42 + run
            result = run_real_channel_reconciliation_trial(bob, alice, bob_ip, qber_i, run, seed,
                                                               reconciliation_prob=prob, k=k_i,
                                                               length_mode="asymptotic")
            result["key_index"] = key_idx
            result["qber"] = qber_i

            row_df = pd.DataFrame([result])
            row_df.to_csv(recon_output_path_multikey, mode="a", header=not header_written, index=False)
            header_written = True

            print(f"  key{key_idx}, prob={prob}, run={run}: non_convergent={result.get('non_convergent')} [saved]")

print("\nMulti-key reconciliation sweep complete.")
print(pd.read_csv(recon_output_path_multikey).groupby(["key_index", "reconciliation_prob"]).size())


## 6. Completeness check

In [ ]:
files_to_check = [
    "sdc_mock_doseresponse_n50.csv",
    "sdc_realchannel_toeplitz_n20_all_modes.csv",
    "sdc_realchannel_finalkey_n20_all_modes.csv",
    "sdc_realchannel_reconciliation_n20_all_modes.csv",
    "sdc_realchannel_verifydigest_n20_all_modes.csv",
    "sdc_real_channel_toeplitz_finalkey_multikey_asymptotic.csv",
    "sdc_real_channel_toeplitz_finalkey_doseresponse_multikey_asymptotic.csv",
    "sdc_real_channel_reconciliation_multikey_asymptotic.csv",
    "key_pairs_metadata.csv",
]

for fname in files_to_check:
    path = PROJECT_DIR / "results" / fname
    if not path.exists():
        print(f"{fname}: NOT FOUND\n")
        continue

    df = pd.read_csv(str(path))
    print(f"=== {fname} ({len(df)} rows) ===")

    if "length_mode" in df.columns:
        print(df.groupby(["length_mode", "prob"]).size().unstack(fill_value=0))
    elif "fault_type" in df.columns and "prob" in df.columns:
        print(df.groupby(["fault_type", "prob"]).size().unstack(fill_value=0))
    elif "prob" in df.columns:
        print(df.groupby("prob").size())

    if "verification_passed" in df.columns:
        n_with_verification = df["verification_passed"].notna().sum()
        print(f"  Rows with verification data: {n_with_verification}/{len(df)}")
        if n_with_verification == 0:
            print("  NOTE: no verification data in this file -- likely run with "
                  "ell=None (no split performed). Do not treat NaN here as "
                  "'verification passed' or 'verification failed'.")
    print()


## 7. Plots: length-mode comparison + undetected-corruption panel (key0, n=20)

In [ ]:
"""
Real Channel (n=20/point/mode): length-mode comparison across all four
fault types, plus an explicit undetected-silent-corruption panel.
"""
import matplotlib.pyplot as plt
from scipy import stats


def wilson_ci(successes, n, confidence=0.95):
    if n == 0:
        return 0.0, 0.0, 0.0
    z = stats.norm.ppf(1 - (1 - confidence) / 2)
    p_hat = successes / n
    denom = 1 + z**2 / n
    center = (p_hat + z**2 / (2 * n)) / denom
    half_width = (z / denom) * np.sqrt(p_hat * (1 - p_hat) / n + z**2 / (4 * n**2))
    return p_hat, max(0, center - half_width), min(1, center + half_width)


def rate_by_prob(df, prob_col, success_col):
    probs = sorted(df[prob_col].unique())
    rates, lo, hi = [], [], []
    for p in probs:
        sub = df[df[prob_col] == p]
        n = len(sub)
        successes = sub[success_col].fillna(False).astype(bool).sum()
        rate, rlo, rhi = wilson_ci(successes, n)
        rates.append(rate); lo.append(rlo); hi.append(rhi)
    return probs, rates, lo, hi


mode_colors = {"placeholder": "tab:blue", "asymptotic": "tab:orange"}

df_toeplitz = pd.read_csv(str(PROJECT_DIR / "results" / "sdc_realchannel_toeplitz_n20_all_modes.csv"))
df_finalkey = pd.read_csv(str(PROJECT_DIR / "results" / "sdc_realchannel_finalkey_n20_all_modes.csv"))
df_recon = pd.read_csv(str(PROJECT_DIR / "results" / "sdc_realchannel_reconciliation_n20_all_modes.csv"))

df_toeplitz["mismatch"] = ~df_toeplitz["keys_match"].fillna(False)
df_finalkey["mismatch"] = ~df_finalkey["keys_match"].fillna(False)

verifydigest_path = PROJECT_DIR / "results" / "sdc_realchannel_verifydigest_n20_all_modes.csv"
df_verifydigest = pd.read_csv(str(verifydigest_path)) if verifydigest_path.exists() else None

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for mode in ["placeholder", "asymptotic"]:
    sub = df_toeplitz[df_toeplitz["length_mode"] == mode]
    probs, rates, lo, hi = rate_by_prob(sub, "prob", "mismatch")
    axes[0, 0].plot(probs, rates, marker='o', label=mode, color=mode_colors[mode])
    axes[0, 0].fill_between(probs, lo, hi, alpha=0.15, color=mode_colors[mode])
axes[0, 0].set_xscale('log'); axes[0, 0].set_xlabel('Fault probability')
axes[0, 0].set_ylabel('Mismatch rate')
axes[0, 0].set_title('Toeplitz-matrix fault')
axes[0, 0].legend(); axes[0, 0].grid(alpha=0.3)

for mode in ["placeholder", "asymptotic"]:
    sub = df_finalkey[df_finalkey["length_mode"] == mode]
    probs, rates, lo, hi = rate_by_prob(sub, "prob", "mismatch")
    axes[0, 1].plot(probs, rates, marker='o', label=mode, color=mode_colors[mode])
    axes[0, 1].fill_between(probs, lo, hi, alpha=0.15, color=mode_colors[mode])
axes[0, 1].set_xscale('log'); axes[0, 1].set_xlabel('Fault probability')
axes[0, 1].set_title('Final-key fault')
axes[0, 1].legend(); axes[0, 1].grid(alpha=0.3)

for mode in ["placeholder", "asymptotic"]:
    sub = df_recon[df_recon["length_mode"] == mode]
    probs, rates, lo, hi = rate_by_prob(sub, "reconciliation_prob", "non_convergent")
    axes[1, 0].plot(probs, rates, marker='o', label=mode, color=mode_colors[mode])
    axes[1, 0].fill_between(probs, lo, hi, alpha=0.15, color=mode_colors[mode])
axes[1, 0].set_xscale('log'); axes[1, 0].set_xlabel('Reconciliation fault probability')
axes[1, 0].set_ylabel('Non-convergence rate')
axes[1, 0].set_title('Reconciliation-state fault')
axes[1, 0].legend(); axes[1, 0].grid(alpha=0.3)

undetected_rows = []
for label, df_src in [("toeplitz", df_toeplitz), ("final_key", df_finalkey)]:
    for mode in ["placeholder", "asymptotic"]:
        sub = df_src[df_src["length_mode"] == mode]
        confirmed_mismatch = sub[sub["mismatch"] == True]
        n_mismatch = len(confirmed_mismatch)
        n_undetected = (confirmed_mismatch["verification_passed"] == True).sum()
        rate = n_undetected / n_mismatch if n_mismatch > 0 else np.nan
        undetected_rows.append({"fault_type": label, "length_mode": mode,
                                  "n_mismatch": n_mismatch, "n_undetected": n_undetected,
                                  "undetected_rate": rate})

df_undetected = pd.DataFrame(undetected_rows)
print("=== Undetected silent-corruption rate (confirmed mismatches only) ===")
print(df_undetected.to_string(index=False))

pivot = df_undetected.pivot(index="fault_type", columns="length_mode", values="undetected_rate")
pivot = pivot[["placeholder", "asymptotic"]]
pivot.plot(kind="bar", ax=axes[1, 1], color=[mode_colors[m] for m in pivot.columns])
axes[1, 1].set_ylabel('Undetected rate (of confirmed mismatches)')
axes[1, 1].set_title('Undetected silent corruption, by length mode')
axes[1, 1].set_xticklabels(axes[1, 1].get_xticklabels(), rotation=0)
axes[1, 1].legend(title="length mode")
axes[1, 1].grid(alpha=0.3, axis='y')

plt.suptitle('Real Channel (n=20/point/mode): Fault Outcomes and Detection Across Length Modes')
plt.tight_layout()
plt.savefig(str(PROJECT_DIR / "results" / "fig_c_lengthmode_comparison_with_verification.png"), dpi=150)
plt.show()

if df_verifydigest is not None:
    print("\n=== verify_digest: false-abort rate (verification_passed=False despite keys_match=True) ===")
    df_verifydigest["false_abort"] = (
        (df_verifydigest["keys_match"] == True) & (df_verifydigest["verification_passed"] == False)
    )
    fa_pivot = df_verifydigest.groupby(["length_mode", "prob"])["false_abort"].mean().unstack(level=0)
    print(fa_pivot.to_string())
else:
    print("\nsdc_realchannel_verifydigest_n20_all_modes.csv not found -- skipping.")
